# Критическая маржа синтетического розничного портфеля

Исследовательский вопрос: какой уровень дополнительной маржи должен обеспечивать более рискованный продукт, чтобы компенсировать кредитный риск относительно ипотеки?

## Метод

Доходная часть строится как `m_i,t = M_t + lambda × (p_i,t − weighted_p_t)`. Риск-скорректированная ставка равна `m_i,t − credit_cost_i,t`, а шестимесячный RAFR — `Exposure_i,t × RAR_i,t × 0.5`. Маржа синтетическая; RAFR не является прибылью ВТБ или RAROC.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root / 'src'))
import stress_model as sm

cfg = sm.load_config(root / 'config/model_config.json')
data = sm.load_dataset(root / cfg.input_file, cfg.data_sheet)
qa = sm.validate_data(data, cfg)
product_results = sm.build_scenarios(data, cfg)
summary = sm.scenario_summary(product_results)
critical = sm.critical_margin_table(product_results, cfg)
stress = sm.stress_critical_margin(product_results, cfg)
mix = sm.portfolio_mix_sensitivity(product_results, cfg)

## Проверка данных

Лист `MODEL_DATA` содержит 24 продуктовых наблюдения за шесть отчётных дат. Ниже проверяются полнота панели и все формулы доходной части вплоть до RAFR.

In [2]:
display(qa)

,check,status,detail
0,duplicate report_date/product,PASS,0
1,complete numeric inputs,PASS,missing_cells=0
2,finite numeric inputs,PASS,all_finite=True
3,positive exposure,PASS,bad_rows=0
4,complete four-product panel,PASS,"dates=6, rows=24"
5,weighted pricing proxy formula,PASS,max_abs_diff=1.388e-17
6,synthetic margin formula,PASS,max_abs_diff=1.388e-17
7,risk-adjusted rate formula,PASS,max_abs_diff=1.388e-17
8,RAFR source formula,PASS,max_abs_diff=8.882e-15
9,baseline products present,PASS,"products=['Auto', 'Cards', 'Consumer', 'Mortga..."


## Сценарии

Moderate увеличивает стоимость кредитного риска на 1 п.п. и снижает якорь маржи на 0,5 п.п. Severe использует сдвиги +3 п.п. и −1 п.п. соответственно.

In [3]:
scenario_view = summary.copy()
scenario_view.columns = ['Сценарий', 'RAFR, млрд руб.']
display(scenario_view.style.format({'RAFR, млрд руб.': '{:.3f}'}))

,Сценарий,"RAFR, млрд руб."
0,Base,20.396
1,Moderate,-28.901
2,Severe,-111.064


## Главный результат: критическая маржа

Критическая маржа выбранного продукта обнуляет совокупный RAFR при неизменных параметрах остальных продуктов. Требуемая премия относительно ипотеки равна критической марже продукта минус текущая синтетическая маржа ипотеки. Отрицательный порог сопровождается отдельным статусом и означает наличие запаса результата у остального портфеля.

In [4]:
critical_view = critical.rename(columns={
    'product': 'Продукт',
    'current_margin': 'Текущая маржа',
    'critical_margin': 'Критическая маржа',
    'required_margin_premium_vs_mortgage': 'Премия к ипотеке',
    'status': 'Статус',
})
display(critical_view.style.format({
    'Текущая маржа': '{:.2%}', 'Критическая маржа': '{:.2%}', 'Премия к ипотеке': '{:.2%}'
}))

,Продукт,Текущая маржа,Критическая маржа,Премия к ипотеке,Статус
0,Mortgage,1.93%,0.98%,0.00%,REFERENCE_PRODUCT
1,Consumer,3.63%,0.96%,-0.97%,FEASIBLE
2,Auto,3.43%,-4.07%,-6.00%,NEGATIVE_THRESHOLD
3,Cards,6.37%,-11.89%,-13.82%,NEGATIVE_THRESHOLD


## Критическая маржа в стрессе

In [5]:
display(stress.style.format({c: '{:.2%}' for c in stress.columns if c.endswith('critical_margin')}))

,product,base_critical_margin,base_status,moderate_critical_margin,moderate_status,severe_critical_margin,severe_status
0,Mortgage,0.98%,REFERENCE_PRODUCT,2.78%,REFERENCE_PRODUCT,6.12%,REFERENCE_PRODUCT
1,Consumer,0.96%,FEASIBLE,6.92%,FEASIBLE,17.19%,FEASIBLE
2,Auto,-4.07%,NEGATIVE_THRESHOLD,13.56%,FEASIBLE,43.27%,FEASIBLE
3,Cards,-11.89%,NEGATIVE_THRESHOLD,31.74%,FEASIBLE,104.80%,FEASIBLE


## Чувствительность к продуктовому составу

Consumer, Auto или Cards увеличиваются за счёт Mortgage при неизменной общей экспозиции. После каждого сдвига пересчитываются средневзвешенный pricing proxy и синтетические маржи.

In [6]:
mix_view = mix.rename(columns={
    'product': 'Продукт', 'share_change_pp': 'Изменение доли, п.п.',
    'critical_margin_change': 'Изменение критической маржи',
    'shifted_critical_margin': 'Новая критическая маржа', 'status': 'Статус'
})
display(mix_view.style.format({
    'Изменение критической маржи': '{:.2%}', 'Новая критическая маржа': '{:.2%}'
}))

,Продукт,"Изменение доли, п.п.",Изменение критической маржи,Новая критическая маржа,Статус
0,Consumer,1.000000,0.24%,1.20%,FEASIBLE
1,Consumer,5.000000,1.02%,1.98%,FEASIBLE
2,Consumer,10.000000,1.71%,2.67%,FEASIBLE
3,Auto,1.000000,1.19%,-2.88%,NEGATIVE_THRESHOLD
4,Auto,5.000000,4.13%,0.06%,FEASIBLE
5,Auto,10.000000,5.96%,1.89%,FEASIBLE
6,Cards,1.000000,5.22%,-6.67%,NEGATIVE_THRESHOLD
7,Cards,5.000000,13.57%,1.68%,FEASIBLE
8,Cards,10.000000,16.85%,4.96%,FEASIBLE


## Выводы и ограничения

В базовом сценарии портфель сохраняет положительный RAFR, а Moderate и Severe пересекают нулевую границу. Стресс резко повышает требуемую критическую маржу, особенно для небольших по экспозиции Auto и Cards. Рост доли рискованного продукта повышает его критическую маржу, поскольку больший объём должен компенсировать возрастающий вклад кредитного риска.

Результаты относятся к синтетическому портфелю. Pricing proxy не является фактической продуктовой доходностью ВТБ; комиссии, операционные расходы, налоги, капитал, RWA и RAROC не моделируются. Шести дат недостаточно для устойчивой статистической оценки.